# 03 - Blocking experiments

This notebook benchmarks the production `blocking.py` implementation. Ground truth is used only to measure candidate recall on training data; it is never used to generate test candidates.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import sys
import time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if not SRC.exists(): SRC = PROJECT_ROOT / 'code' / 'business_entity_resolution' / 'src'
sys.path.insert(0, str(SRC.resolve()))
from config import default_config
from data_loader import load_ground_truth, load_training_data
from preprocessing import preprocess_sources
from blocking import BlockingResult, generate_candidates, evaluate_blocking_recall

CONFIG = default_config().resolved(PROJECT_ROOT)
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports' / 'blocking'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MAX_REFERENCE_ROWS = None  # Set a positive value for a faster exploratory sample.


In [ ]:
training = load_training_data(CONFIG)
ground_truth = training.ground_truth
sources = preprocess_sources(training.sources, CONFIG.schema, CONFIG.normalization)
source_names = list(sources)
if len(source_names) < 2: raise ValueError('At least one reference and one candidate source are required')
reference_source = source_names[0]
reference = sources[reference_source]
candidate_sources = {name: sources[name] for name in source_names[1:]}
if MAX_REFERENCE_ROWS is not None: reference = reference.head(MAX_REFERENCE_ROWS).copy()
print('Reference source:', reference_source, 'rows:', len(reference))
print('Candidate sources:', {name: len(frame) for name, frame in candidate_sources.items()})

In [ ]:
truth = {str(row[CONFIG.schema.ground_truth_source_id_column]): set(filter(None, str(row[CONFIG.schema.ground_truth_matches_column] or '').split(','))) for _, row in ground_truth.iterrows()}
truth = {key: value for key, value in truth.items() if key in set(reference[CONFIG.schema.entity_id_column].astype(str))}
universe_size = sum(len(frame) for frame in candidate_sources.values())
def measured_metrics(result, method, elapsed):
    pairs = result.pairs
    grouped = pairs.groupby('reference_entity_id')['candidate_entity_id'].nunique() if not pairs.empty else pd.Series(dtype='int64')
    found = {str(ref): set(group['candidate_entity_id'].astype(str)) for ref, group in pairs.groupby('reference_entity_id')} if not pairs.empty else {}
    total_true = sum(len(values) for values in truth.values())
    recovered = sum(len(values & found.get(ref, set())) for ref, values in truth.items())
    false_negatives = total_true - recovered
    candidate_count = len(pairs)
    universe = max(1, len(reference) * universe_size)
    return {'method': method, 'candidate_count': candidate_count, 'candidate_recall': recovered / total_true if total_true else 1.0, 'false_negative_count': false_negatives, 'average_candidates_per_reference': candidate_count / max(1, len(reference)), 'median_candidates_per_reference': float(grouped.median()) if len(grouped) else 0.0, 'maximum_candidates_per_reference': int(grouped.max()) if len(grouped) else 0, 'reduction_ratio': 1 - candidate_count / universe, 'runtime_seconds': elapsed, 'references_with_candidates': int(len(grouped))}


In [ ]:
method_settings = {
    'exact_name': {'enabled_methods': ('normalized_name_exact',)},
    'name_tokens': {'enabled_methods': ('name_token',)},
    'name_prefix': {'enabled_methods': ('name_prefix',)},
    'address_tokens': {'enabled_methods': ('address_token',)},
    'postal_tokens': {'enabled_methods': ('postal_token',)},
    'locality_tokens': {'enabled_methods': ('locality_token',)},
    'character_ngrams': {'enabled_methods': ('name_ngram', 'address_ngram'), 'use_character_ngrams': True},
    'phonetic': {'enabled_methods': ('name_phonetic',), 'use_phonetic_keys': True},
    'composite': {'enabled_methods': ('country_name', 'country_address'), 'use_combined_keys': True},
}
results = []
runs = {}
for method, changes in method_settings.items():
    settings = replace(CONFIG.blocking, **changes)
    started = time.perf_counter()
    result = generate_candidates(reference, candidate_sources, CONFIG, reference_source=reference_source, settings=settings)
    elapsed = time.perf_counter() - started
    runs[method] = result
    results.append(measured_metrics(result, method, elapsed))
comparison = pd.DataFrame(results).sort_values(['candidate_recall', 'reduction_ratio'], ascending=[False, False], kind='mergesort')
display(comparison)

In [ ]:
# Complementary union experiment using the production default multi-stage configuration.
started = time.perf_counter()
union_result = generate_candidates(reference, candidate_sources, CONFIG, reference_source=reference_source)
union_elapsed = time.perf_counter() - started
union_metrics = measured_metrics(union_result, 'configured_union', union_elapsed)
display(pd.DataFrame([union_metrics]))

# Unique true links recovered by the union but missed by each individual method.
union_found = {str(ref): set(group['candidate_entity_id'].astype(str)) for ref, group in union_result.pairs.groupby('reference_entity_id')}
unique_rows = []
for method, result in runs.items():
    method_found = {str(ref): set(group['candidate_entity_id'].astype(str)) for ref, group in result.pairs.groupby('reference_entity_id')}
    unique_rows.append({'method': method, 'true_links_union_recovers_but_method_misses': sum(len((truth.get(ref, set()) & union_found.get(ref, set())) - method_found.get(ref, set())) for ref in truth)})
display(pd.DataFrame(unique_rows))

In [ ]:
# Diagnostics by country and singleton status use ground truth only for measurement.
ref_country = reference.set_index(CONFIG.schema.entity_id_column)[CONFIG.schema.country_column].astype(str).to_dict()
singleton_ids = {ref for ref, values in truth.items() if not values}
country_rows = []
for country in sorted(set(ref_country.values())):
    refs = {ref for ref, value in ref_country.items() if value == country}
    recovered = sum(len(truth.get(ref, set()) & union_found.get(ref, set())) for ref in refs)
    actual = sum(len(truth.get(ref, set())) for ref in refs)
    country_rows.append({'country': country, 'reference_count': len(refs), 'true_link_count': actual, 'candidate_recall': recovered / actual if actual else 1.0})
country_diagnostics = pd.DataFrame(country_rows)
singleton_candidates = sum(bool(union_found.get(ref, set())) for ref in singleton_ids)
display(country_diagnostics)
print('Singleton references:', len(singleton_ids), 'with candidates:', singleton_candidates)

In [ ]:
comparison.to_csv(REPORT_DIR / 'blocking_comparison.csv', index=False)
pd.DataFrame([union_metrics]).to_csv(REPORT_DIR / 'blocking_results.csv', index=False)
country_diagnostics.to_csv(REPORT_DIR / 'country_diagnostics.csv', index=False)

selected = union_metrics
summary = f'''# Blocking experiment summary\n\nResults were measured on the configured training reference and candidate sources. Ground truth was used only to measure recall.\n\n- Selected configured strategy: multi-stage union from `CONFIG.blocking`.\n- Candidate recall: {selected['candidate_recall']:.6f}.\n- Candidate count: {selected['candidate_count']}.\n- Average candidates per reference: {selected['average_candidates_per_reference']:.4f}.\n- Median candidates per reference: {selected['median_candidates_per_reference']:.4f}.\n- Maximum candidates per reference: {selected['maximum_candidates_per_reference']}.\n- Reduction ratio: {selected['reduction_ratio']:.6f}.\n- False-negative candidate count: {selected['false_negative_count']}.\n- Runtime seconds: {selected['runtime_seconds']:.4f}.\n\n## Selection rule\n\nThe final strategy must prioritize candidate recall, then use candidate volume, reduction ratio, runtime, and country/singleton diagnostics as safety checks. No candidates were manually inserted. The exact active methods are recorded in the configuration used for this run.\n'''
(REPORT_DIR / 'blocking_summary.md').write_text(summary, encoding='utf-8')
print('Saved blocking reports to', REPORT_DIR)

## Final selection protocol

Select the production blocker only after comparing `blocking_comparison.csv` and `blocking_results.csv`. The chosen configuration must retain the measured high-recall union, preserve open-set country handling, and keep the final candidate table equal to the last set passed to pairwise feature/model scoring.